In [1]:
import pandas as pd
from datetime import datetime
from io import BytesIO
from google.cloud import storage, bigquery
import numpy as np
import warnings
pd.set_option('display.max_columns', None)
warnings.filterwarnings("ignore", category=UserWarning, module="openpyxl")

### Reporte Emision Acsel X

In [60]:
df_rep_emision_acselX= pd.read_csv('C:/data/Reportes de Emision/PACTIVAS_63886903_01_9013.TXT', sep="®", dtype=str, encoding="latin1", engine="python")
df_rep_emision_acselX.columns = df_rep_emision_acselX.columns.str.strip().str.upper().str.replace(r'[^A-Za-z0-9]', '_', regex=True)

In [56]:
df_rep_emision_acselX.rename(columns={
    'FECMOV': 'FECHA_MOVIMIENTO',  'TIPOOP': 'TIPO_OPERACION',  'CODPROD': 'CODIGO_PRODUCTO_AX',  
    'CODPOL': 'CODIGO_POLIZA',  'NUMPOL': 'NRO_POLIZA', 'NUMCERT': 'NRO_CERT',  'STSCERT': 'ESTADO_CERT',
    'CODMONEDA': 'MONEDA',  'MTOOPER': 'MONTO_OPERACION', 'NUMOPER': 'NRO_OPERACION'
}, inplace=True)

In [57]:
df_rep_emision_acselX= df_rep_emision_acselX[['FECHA_MOVIMIENTO','TIPO_OPERACION','CODIGO_PRODUCTO_AX','CODIGO_POLIZA',
                                            'NRO_POLIZA','NRO_CERT','ESTADO_CERT','MONEDA','MONTO_OPERACION','NRO_OPERACION']]

In [58]:
df_rep_emision_acselX['FECHA_MOVIMIENTO'] = pd.to_datetime(df_rep_emision_acselX['FECHA_MOVIMIENTO'], format="%d/%m/%Y", errors='coerce').dt.date
df_rep_emision_acselX['MONTO_OPERACION'] = pd.to_numeric(df_rep_emision_acselX['MONTO_OPERACION'], errors="coerce").astype('float64')

In [61]:
df_rep_emision_acselX.head(3)

,PRODUCCION,FECMOV,HORAMOV,TIPOOP,CODOFISUSC,DESOFISUSC,CODPROD,DESCPROD,CODPOL,NUMPOL,NUMCERT,STSCERT,CODPLAN,REVPLAN,DESCPLANPROD,NUMID,DESCCERT,FECEMI,FECINI,FECFIN,CODMONEDA,MTOOPER,MONTO_ANULADO,CODINTER,DESCINTER,NOMUSER,IDEPOL,NUMTRAMITE,NUMOPER,CODOFIEMI,DESCOFIEMI,ENLACE,MOTIVOEND,DESCRESPAGO,TIPOPAGO,CNOMPLAN,NUMDOC,FECVTO,INDLICIT,MONEDA_PRIMA_NETA_AL__RIMAC,PRIMA_NETA_AL__RIMAC
0,RIMAC,25/03/2026,14:31,MOD,100000,AGENCIA SAN ISIDRO,9013,XXXX,9013,500000,2160,ACT,001,001,ACCIDENTES PERSONALES Y ASISTENCIAS BBVA,00000006080819,NELLY SANTA BALTAZAR PAUCAR,21/02/2008,01/05/2009,19/08/2028,SOL,8.08,0,43,SEGUROS DIRECTOS,NATALIA ELIZABETH AGUILAR ZUMAETA,2614524,SIN TRAMITE,4743288578,100000,AGENCIA SAN ISIDRO,NaN,NaN,BANCO BBVA PERU,CONTADO,CON002,1176821456,24/04/2026,N,NaN,0
1,RIMAC,25/03/2026,14:31,MOD,100000,AGENCIA SAN ISIDRO,9013,XXXX,9013,500000,2180,ACT,001,001,ACCIDENTES PERSONALES Y ASISTENCIAS BBVA,00000005436002,YUDITH TOMAS MOLINA,21/02/2008,01/05/2009,19/08/2028,SOL,8.08,0,43,SEGUROS DIRECTOS,NATALIA ELIZABETH AGUILAR ZUMAETA,2614524,SIN TRAMITE,4743288578,100000,AGENCIA SAN ISIDRO,NaN,NaN,BANCO BBVA PERU,CONTADO,CON002,1176821457,24/04/2026,N,NaN,0
2,RIMAC,25/03/2026,14:31,MOD,100000,AGENCIA SAN ISIDRO,9013,XXXX,9013,500000,2335,ACT,001,001,ACCIDENTES PERSONALES Y ASISTENCIAS BBVA,00000006372695,ELSA CAROLINA JUMPA VALLADARES,21/02/2008,01/05/2009,19/08/2028,SOL,8.08,0,43,SEGUROS DIRECTOS,NATALIA ELIZABETH AGUILAR ZUMAETA,2614524,SIN TRAMITE,4743288578,100000,AGENCIA SAN ISIDRO,NaN,NaN,BANCO BBVA PERU,CONTADO,CON002,1176821467,24/04/2026,N,NaN,0


### Reporte Emision SAS

In [ ]:
df_reporte_emision_SAS= pd.read_csv('c:/data/Reportes de Emision/PM MARZO 2503 DOLARES_97992.csv', sep=',', encoding='latin-1', dtype=str, skiprows=7)
df_reporte_emision_SAS.columns = df_reporte_emision_SAS.columns.str.strip().str.upper().str.replace(r'[^A-Za-z0-9]', '_', regex=True)

In [47]:
df_reporte_emision_SAS.rename(columns={
    'C_DIGO_PRODUCTO': 'CODIGO_PRODUCTO_SAS','C_D__CERTIFICADO_CANAL': 'CERTIFICADO_CANAL',  
    'FEC__INICIO': 'FECHA_INICIO', 'FEC__FIN': 'FECHA_FIN', 'LINEA_TRAMA_TEXTO_COMPLETO_': 'LINEA_TRAMA',
    'DESCRIPCI_N_DE_ESTADO': 'DESCRIPCION_ESTADO', 'NUMERO_DE_POLIZA': 'NRO_POLIZA', 
    'TIPO_DOC_EMISI_N': 'TIPO_DOC_EMISION', 'N__DOC_EMISI_N': 'NRO_DOC_EMISION', 'FECHA_DE_EMISI_N': 'FECHA_EMISION',  
    'ESTADO_DE_DOCUMENTO': 'ESTADO_DOC'
}, inplace=True)

In [48]:
df_reporte_emision_SAS= df_reporte_emision_SAS[['CODIGO_PRODUCTO_SAS','CERTIFICADO_CANAL','FECHA_INICIO','FECHA_FIN',
                                            'TIPO_MOVIMIENTO','MONEDA','PRIMA_BRUTA','DESCRIPCION_ESTADO','NRO_POLIZA',
                                            'NRO_DOC_EMISION', 'TIPO_DOC_EMISION','FECHA_EMISION','ESTADO_DOC','LINEA_TRAMA']]

In [50]:
df_reporte_emision_SAS['CERTIFICADO_CANAL']= df_reporte_emision_SAS['CERTIFICADO_CANAL'].str[1:]
df_reporte_emision_SAS['NRO_POLIZA']= df_reporte_emision_SAS['NRO_POLIZA'].str[1:]
df_reporte_emision_SAS['FECHA_INICIO'] = pd.to_datetime(df_reporte_emision_SAS['FECHA_INICIO'], format="%d/%m/%Y", errors='coerce').dt.date
df_reporte_emision_SAS['FECHA_FIN'] = pd.to_datetime(df_reporte_emision_SAS['FECHA_FIN'], format="%d/%m/%Y", errors='coerce').dt.date
df_reporte_emision_SAS['FECHA_EMISION'] = pd.to_datetime(df_reporte_emision_SAS['FECHA_EMISION'], format='%d-%b-%y', errors='coerce').dt.date
df_reporte_emision_SAS['PRIMA_BRUTA'] = pd.to_numeric(df_reporte_emision_SAS['PRIMA_BRUTA'], errors="coerce").astype('float64')

In [51]:
df_reporte_emision_SAS.head(3)

,CODIGO_PRODUCTO_SAS,CERTIFICADO_CANAL,FECHA_INICIO,FECHA_FIN,TIPO_MOVIMIENTO,MONEDA,PRIMA_BRUTA,DESCRIPCION_ESTADO,NRO_POLIZA,NRO_DOC_EMISION,TIPO_DOC_EMISION,FECHA_EMISION,ESTADO_DOC,LINEA_TRAMA
0,8018,00110465254000912985,2026-02-28,2026-03-31,Renovacion,USD,10.41,Emitido en A/X,500002,1175995167,LQ,2026-03-03,COB,803001104652540009129850465 01U...
1,8018,00110465254001819482,2026-03-01,2026-04-01,Renovacion,USD,12.39,Emitido en A/X,500006,1175994398,LQ,2026-03-03,COB,803001104652540018194820465 01U...
2,8018,00110465264001470293,2026-03-02,2026-04-02,Renovacion,USD,10.41,Emitido en A/X,500006,1175994380,LQ,2026-03-03,COB,803001104652640014702930465 01U...


In [53]:
df_reporte_emision_SAS.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1068 entries, 0 to 1067
Data columns (total 14 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   CODIGO_PRODUCTO_SAS  1068 non-null   object 
 1   CERTIFICADO_CANAL    1068 non-null   object 
 2   FECHA_INICIO         1068 non-null   object 
 3   FECHA_FIN            1068 non-null   object 
 4   TIPO_MOVIMIENTO      1068 non-null   object 
 5   MONEDA               1068 non-null   object 
 6   PRIMA_BRUTA          1021 non-null   float64
 7   DESCRIPCION_ESTADO   1068 non-null   object 
 8   NRO_POLIZA           1068 non-null   object 
 9   NRO_DOC_EMISION      1021 non-null   object 
 10  TIPO_DOC_EMISION     1021 non-null   object 
 11  FECHA_EMISION        1021 non-null   object 
 12  ESTADO_DOC           1068 non-null   object 
 13  LINEA_TRAMA          1068 non-null   object 
dtypes: float64(1), object(13)
memory usage: 116.9+ KB
